# Deep Agents + Elasticsearch: Systematic Research with Multi-Agent Evaluation
;
**LangChain** provides composable building blocks for LLM applications: chains, prompts, and tool integrations. **LangGraph** adds stateful graph-based workflows with branching and cycles. **Deep Agents** build on top of LangGraph and add built-in middleware for **planning** (TODO lists), **file I/O**, and **sub-agent delegation**, capabilities you'd otherwise wire manually.

This notebook uses Deep Agents to run a **systematic literature review** where specialized sub-agents independently research, evaluate, and cross-check findings storing structured results in **Elasticsearch** for hybrid search and analysis.

## Why Deep Agents for This Use Case

| Framework | Best For | Example |
|-----------|----------|---------|
| **LangChain** | Simple chains: prompt > LLM > output | Summarize a document, answer a question |
| **LangGraph** | Custom stateful workflows with branching and cycles | Multi-step approval pipelines, chatbots with tool use |
| **Deep Agents** | Long-running autonomous research with planning, sub-agents, and memory | Systematic review with multi-agent evaluation |

A simple agent with tools could search and store findings. But a **systematic review** requires the orchestrator to **plan research angles**, **delegate each angle to a specialized sub-agent**, and then **cross-evaluate** findings for credibility and consensus, exactly the pattern Deep Agents automate with built-in TODO lists and sub-agent middleware.

## Setup

In [28]:
%pip install deepagents langchain-elasticsearch langchain-google-genai python-dotenv tavily-python -q


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: /opt/homebrew/opt/python@3.11/bin/python3.11 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [29]:
import os
from dotenv import load_dotenv

load_dotenv()

ELASTICSEARCH_URL = os.getenv("ELASTICSEARCH_URL")
ELASTICSEARCH_API_KEY = os.getenv("ELASTICSEARCH_API_KEY")
KIBANA_URL = os.getenv("KIBANA_URL")
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

## Initialize Elasticsearch Client

In [30]:
from elasticsearch import Elasticsearch

es_client = Elasticsearch(
    ELASTICSEARCH_URL,
    api_key=ELASTICSEARCH_API_KEY,
)

es_client.info()

ObjectApiResponse({'name': 'instance-0000000004', 'cluster_name': 'fd31ad5f77d841d69b622c17ed64b766', 'cluster_uuid': 'YxmDkf9DSwWpvcCLv98q8A', 'version': {'number': '9.3.0', 'build_flavor': 'default', 'build_type': 'docker', 'build_hash': '17b451d8979a29e31935fe1eb901310350b30e62', 'build_date': '2026-01-29T10:05:46.708397977Z', 'build_snapshot': False, 'lucene_version': '10.3.2', 'minimum_wire_compatibility_version': '8.19.0', 'minimum_index_compatibility_version': '8.0.0'}, 'tagline': 'You Know, for Search'})

## Define the Research Index

Each finding includes structured evaluation metadata: `research_angle`, `evidence_type`, `relevance_score`, and `source_credibility`. Text fields use `copy_to` to populate a `semantic_text` field backed by Jina Embeddings v5.

In [31]:
INDEX_NAME = "deep-agent-research"
INFERENCE_ID = ".jina-embeddings-v5-text-small"

index_body = {
    "mappings": {
        "properties": {
            "query": {"type": "text", "copy_to": "semantic_field"},
            "source": {"type": "keyword"},
            "title": {
                "type": "text",
                "fields": {"keyword": {"type": "keyword"}},
                "copy_to": "semantic_field",
            },
            "content": {"type": "text", "copy_to": "semantic_field"},
            "timestamp": {"type": "date"},
            "tags": {"type": "keyword"},
            "research_angle": {"type": "keyword"},
            "evidence_type": {"type": "keyword"},
            "relevance_score": {"type": "float"},
            "source_credibility": {"type": "keyword"},
            "semantic_field": {
                "type": "semantic_text",
                "inference_id": INFERENCE_ID,
            },
        }
    }
}

if not es_client.indices.exists(index=INDEX_NAME):
    es_client.indices.create(index=INDEX_NAME, body=index_body)
    print(f"Created index: {INDEX_NAME}")
else:
    print(f"Index already exists: {INDEX_NAME}")

Index already exists: deep-agent-research


## Build Custom Tools

1. **web_search** - search the web via [Tavily](https://docs.tavily.com/documentation/integrations/langchain)
2. **store_finding** - save a structured finding to Elasticsearch
3. **query_elasticsearch** - let the agent explore stored findings (search, aggregate, filter)

In [32]:
import json
from datetime import datetime, timezone

from langchain_core.tools import tool
from tavily import TavilyClient

tavily = TavilyClient(api_key=TAVILY_API_KEY)


@tool
def web_search(query: str) -> str:
    """Search the web for information on a topic."""

    results = tavily.search(query=query, max_results=5)
    formatted = []

    for r in results["results"]:
        formatted.append(f"**{r['title']}**\nURL: {r['url']}\n{r['content'][:500]}")

    return "\n\n---\n\n".join(formatted)


@tool
def store_finding(
    title: str,
    source: str,
    content: str,
    tags: list[str],
    query: str,
    research_angle: str,
    evidence_type: str,
    relevance_score: float,
    source_credibility: str,
) -> str:
    """Store a research finding in Elasticsearch with structured metadata.

    Args:
        title: Short title for the finding.
        source: URL or citation.
        content: Summary of the finding.
        tags: Keyword tags.
        query: The original research query.
        research_angle: Which angle this covers (e.g. "performance", "cost", "integration").
        evidence_type: Type of evidence (e.g. "benchmark", "case_study", "peer_reviewed", "expert_opinion").
        relevance_score: How relevant this is to the query (1.0 to 10.0).
        source_credibility: Credibility level ("peer_reviewed", "preprint", "industry_report", "blog", "documentation").
    """

    doc = {
        "query": query,
        "source": source,
        "title": title,
        "content": content,
        "tags": tags,
        "research_angle": research_angle,
        "evidence_type": evidence_type,
        "relevance_score": relevance_score,
        "source_credibility": source_credibility,
        "timestamp": datetime.now(timezone.utc).isoformat(),
    }

    resp = es_client.index(index=INDEX_NAME, document=doc)

    return f"Stored finding '{title}' (id: {resp['_id']})"


@tool
def query_elasticsearch(query_body: str) -> str:
    """Run an Elasticsearch query and return results. Accepts a JSON string with a valid Elasticsearch query body.

    Used by the evaluator and reviewer sub-agents to analyze stored findings during the review phases.

    Use this to search, filter, or aggregate stored findings. Examples:
    - Match all: {"query": {"match_all": {}}, "size": 5}
    - Aggregate by tag: {"size": 0, "aggs": {"by_tag": {"terms": {"field": "tags"}}}}
    - Filter by angle: {"query": {"term": {"research_angle": "performance"}}}
    - Average score by angle: {"size": 0, "aggs": {"by_angle": {"terms": {"field": "research_angle"}, "aggs": {"avg_score": {"avg": {"field": "relevance_score"}}}}}}
    """

    es_client.indices.refresh(index=INDEX_NAME)
    body = json.loads(query_body)
    response = es_client.search(index=INDEX_NAME, body=body)

    return json.dumps(response.body, default=str)

## Create the Deep Agent with Sub-Agents

The orchestrator delegates work to three specialized sub-agents:
- **angle-researcher**: searches the web and stores structured findings per research angle
- **finding-evaluator**: queries Elasticsearch to score and assess stored findings
- **cross-reviewer**: compares findings across angles, flags contradictions, and scores consensus

The orchestrator uses the built-in `write_todos` tool to plan which angles to investigate before delegating.

> **Note:** Preview models expire periodically. Check the [available models list](https://ai.google.dev/gemini-api/docs/models) and use a recent one.

In [33]:
from deepagents import create_deep_agent
from langchain_google_genai import ChatGoogleGenerativeAI

model = ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite-preview", temperature=0)

# Sub-agent: researches a specific angle and stores findings
angle_researcher = {
    "name": "angle-researcher",
    "description": (
        "Researches a specific angle of a research query. "
        "Searches the web, evaluates sources, and stores each finding "
        "in Elasticsearch with structured metadata."
    ),
    "system_prompt": (
        "You research ONE specific angle of a scientific query.\n"
        "1. Use web_search to find 3-5 relevant sources for the given angle.\n"
        "2. For EACH useful source, call store_finding with:\n"
        "   - research_angle: the angle you were assigned\n"
        "   - evidence_type: 'benchmark', 'case_study', 'peer_reviewed', or 'expert_opinion'\n"
        "   - relevance_score: 1.0-10.0 based on how directly it addresses the query\n"
        "   - source_credibility: 'peer_reviewed', 'preprint', 'industry_report', 'blog', or 'documentation'\n"
        "3. Return a brief summary of what you found for this angle."
    ),
    "tools": [web_search, store_finding],
}

# Sub-agent: evaluates stored findings using ES queries
finding_evaluator = {
    "name": "finding-evaluator",
    "description": (
        "Evaluates and analyzes findings already stored in Elasticsearch. "
        "Aggregates by research angle, computes average scores, and identifies "
        "which angles have the strongest evidence."
    ),
    "system_prompt": (
        "You analyze research findings stored in Elasticsearch.\n"
        "Use query_elasticsearch to:\n"
        "1. Aggregate findings by research_angle and compute avg relevance_score.\n"
        "2. Aggregate by source_credibility to assess evidence quality.\n"
        "3. Aggregate by evidence_type to understand the mix of evidence.\n"
        "4. Identify which angles have the most and strongest findings.\n"
        "Return a structured evaluation summary."
    ),
    "tools": [query_elasticsearch],
}

# Sub-agent: cross-checks findings for contradictions and consensus
cross_reviewer = {
    "name": "cross-reviewer",
    "description": (
        "Reviews all stored findings to identify contradictions, consensus, "
        "and gaps across research angles. Produces a final assessment."
    ),
    "system_prompt": (
        "You are a critical reviewer of research findings.\n"
        "Use query_elasticsearch to retrieve all findings, then:\n"
        "1. Identify claims that appear across multiple angles (consensus).\n"
        "2. Flag any contradictions between sources.\n"
        "3. Note which angles lack peer-reviewed evidence.\n"
        "4. Produce a final ranked recommendation based on the evidence.\n"
        "Be skeptical. Prioritize peer-reviewed sources over blogs."
    ),
    "tools": [query_elasticsearch],
}

agent = create_deep_agent(
    model=model,
    tools=[web_search, store_finding, query_elasticsearch],
    subagents=[angle_researcher, finding_evaluator, cross_reviewer],
    system_prompt=(
        "You are a systematic research orchestrator. For each query:\n\n"
        "1. PLAN: Use write_todos to break the query into 3-4 research angles "
        "(e.g., 'performance benchmarks', 'cost and accessibility', 'domain-specific capabilities').\n\n"
        "2. RESEARCH: For each angle, delegate to the 'angle-researcher' sub-agent "
        "using the task tool. Each sub-agent call should specify the angle and original query.\n\n"
        "3. EVALUATE: Delegate to 'finding-evaluator' to analyze the stored findings "
        "and assess evidence quality across angles.\n\n"
        "4. REVIEW: Delegate to 'cross-reviewer' to cross-check findings, flag "
        "contradictions, and produce a consensus score.\n\n"
        "5. SYNTHESIZE: Write a final report summarizing the best answer based on "
        "the evidence, noting confidence levels and gaps.\n\n"
        "Mark each TODO as done as you complete it."
    ),
)

## Run Research Query

The orchestrator will plan angles, delegate to sub-agents, evaluate, and synthesize.

In [34]:
research_query = (
    "What is the best LLM for biochemical analysis of cancer cells in mice?"
)

result = agent.invoke(
    {"messages": [{"role": "user", "content": research_query}]},
)

print(result["messages"][-1].content)

KeyboardInterrupt: 

In [24]:
second_query = "What is the best front-end framework to connect to a DeepAgent service for assigning tasks and generating research documents?"

result2 = agent.invoke(
    {"messages": [{"role": "user", "content": second_query}]},
)

print(result2["messages"][-1].content)

[{'type': 'text', 'text': 'The best front-end framework for connecting to a DeepAgent service depends on your team\'s expertise and the complexity of your UI requirements. Based on current industry standards for AI-driven applications, here is the breakdown:\n\n### Top Recommendation: Next.js (React)\nNext.js is currently the industry standard for AI-driven applications, primarily due to the **Vercel AI SDK**.\n\n*   **Why it\'s the best:**\n    *   **Generative UI:** The Vercel AI SDK allows the LLM to stream entire UI components (e.g., task cards, document editors, charts) directly to the client. This is ideal for complex, multi-step workflows.\n    *   **Streaming Support:** It provides first-class support for Server-Sent Events (SSE) and streaming text/structured data, which is essential for real-time agent feedback.\n    *   **Ecosystem:** The ecosystem for rendering Markdown, LaTeX, and complex document structures is the most mature in the React space.\n    *   **State Management

## Explore Findings with Elastic Agent Builder

Instead of writing Elasticsearch queries by hand, we create an agent via the [Agent Builder API](https://www.elastic.co/docs/explore-analyze/ai-features/agent-builder/kibana-api) and ask it questions about our stored research.

In [ ]:
import requests

headers = {
    "Authorization": f"ApiKey {ELASTICSEARCH_API_KEY}",
    "Content-Type": "application/json",
    "kbn-xsrf": "true",
}

# Create an ES|QL tool that queries our research index
tool_body = {
    "id": "research-findings-tool",
    "type": "esql",
    "description": (
        "Query the deep-agent-research index to explore stored findings. "
        "Returns average relevance scores and counts grouped by research angle and source credibility."
    ),
    "configuration": {
        "query": (
            "FROM deep-agent-research "
            "| STATS avg_score = AVG(relevance_score), count = COUNT(*) BY research_angle, source_credibility"
        ),
    },
}

resp = requests.post(
    f"{KIBANA_URL}/api/agent_builder/tools", headers=headers, json=tool_body
)
print(resp.json())

# Create the Agent Builder agent
agent_body = {
    "id": "research-explorer",
    "name": "Research Explorer",
    "description": "Explores and analyzes stored research findings from the deep-agent-research index.",
    "configuration": {
        "instructions": (
            "You help users explore research findings stored in Elasticsearch. "
            "Use the research-findings-tool to query findings by angle, "
            "summarize evidence quality, and answer questions about the stored research."
        ),
        "tools": [{"tool_ids": ["research-findings-tool"]}],
    },
}

resp = requests.post(
    f"{KIBANA_URL}/api/agent_builder/agents", headers=headers, json=agent_body
)
print(resp.json())

Tool created: 400
{'statusCode': 400, 'error': 'Bad Request', 'message': 'Tool with id research-findings-tool already exists', 'attributes': {}}
Agent created: 400
{'statusCode': 400, 'error': 'Bad Request', 'message': 'Agent with id research-explorer already exists', 'attributes': {}}


In [39]:
# Ask the agent about our findings
converse_body = {
    "agent_id": "research-explorer",
    "input": "Which research angle has the highest average relevance score? Summarize the evidence quality.",
}

resp = requests.post(
    f"{KIBANA_URL}/api/agent_builder/converse", headers=headers, json=converse_body
)

conversation = resp.json()
print(conversation["response"]["message"])

Based on my search of the research database, I'm unable to determine which research angle has the highest average relevance score. The database queries for various potential research angles including performance, cost, effectiveness, security, reliability, usability, innovation, ethics, and scalability did not return any results.

This could indicate that:
- The research database may not contain any entries at this time
- The research angles may be categorized differently than the terms searched
- There might be an issue with the data structure or indexing

Without access to the relevance score data, I cannot provide a summary of evidence quality across the different research angles. If you'd like to explore this further, you might consider checking if the database has been populated with research findings or if there are alternative terms used for categorizing the research angles.


In [40]:
# Follow-up question in the same conversation
converse_body = {
    "agent_id": "research-explorer",
    "conversation_id": conversation.get("conversation_id"),
    "input": "What are the most common tags across findings? Are there any angles with no peer-reviewed sources?",
}

resp = requests.post(
    f"{KIBANA_URL}/api/agent_builder/converse", headers=headers, json=converse_body
)

print(resp.json()["response"]["message"])

Based on my search of the research database, I'm unable to determine the most common tags across findings or identify angles without peer-reviewed sources. The database appears to be empty or not properly configured, as multiple queries across different research angles did not return any results.

This suggests that either:

1. The research database hasn't been populated with findings yet
2. There may be an issue with how the data is indexed or structured
3. The research findings might be stored using different terminology or categorization than what was queried

Without access to the underlying data, I cannot provide information about tag frequency or the peer review status across different research angles. To proceed with this analysis, the database would need to be populated with research findings or the query approach would need to be adjusted to match how the data is actually stored.


## Cleanup

In [ ]:
# Delete Agent Builder resources
requests.delete(
    f"{KIBANA_URL}/api/agent_builder/agents/research-explorer", headers=headers
)
requests.delete(
    f"{KIBANA_URL}/api/agent_builder/tools/research-findings-tool", headers=headers
)
print("Deleted Agent Builder agent and tool.")

In [19]:
# Delete the research index
es_client.indices.delete(index=INDEX_NAME)

ObjectApiResponse({'acknowledged': True})